# M3 · Llantas con keypoints (YOLO11s-pose) y conteo de ejes con dataset público
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**
Basado en `keypoints_llantas_yolo_detectron2_v3.ipynb` (Ing. Alvaro Pinto Mamani): mismo orden de keypoints y
misma regla de conteo de ejes que el servidor (`inferencia/peaje_ia/m3_ejes.py`).

| | |
|---|---|
| **Dataset** | [*axle*](https://universe.roboflow.com/eternal-wiecw/axle-bjpov) — autor **eternal**, Roboflow Universe |
| **Licencia** | **La página de Roboflow Universe no muestra licencia**: se cita al autor y se documenta así en `entrenamiento/DATASETS.md` |
| **Versión / formato** | v5 · `yolov11` (detección: cajas) · clases `]`, `axle`, `B` → solo se usa **`axle`** |
| **Keypoints** | **0 centro · 1 contacto · 2 superior · 3 izquierda · 4 derecha** (`kpt_shape [5, 3]`, `flip_idx [0, 1, 2, 4, 3]`) |
| **Salida** | `m3_llantas_pose.pt` → `MyDrive/peaje-ia/pesos/` → `inferencia/pesos/` del Pod |
| **Métricas** | mAP50 de pose y de caja + **MAE del conteo de ejes** en el split test |

> ### ⚠️ Limitación declarada: keypoints derivados geométricamente
> El dataset solo trae **cajas** de llantas/ejes. Los 5 keypoints de cada llanta **no fueron etiquetados a
> mano**: se derivan de la caja (centro, punto medio inferior = contacto, punto medio superior, extremos
> izquierdo y derecho). Por eso el mAP de pose mide qué tan bien el modelo reproduce esa geometría, no
> keypoints anatómicos reales. Las llantas vistas en perspectiva (elipses inclinadas) quedan aproximadas.
> El MAE de ejes sí es una métrica real del sistema: compara el número de cajas `axle` por imagen con el
> conteo que produce el servidor a partir de los keypoints predichos. En `m3_metricas.json` queda el campo
> `keypoints_origen: "derivados geométricamente de cajas del dataset eternal/axle"`.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "ACTIVE LA GPU T4"
!pip -q install ultralytics==8.4.218 roboflow
import os, sys, json, glob, time, shutil, subprocess, yaml, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO (active la T4)')

## 2. Configuración (editable)

In [ ]:
RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO = 'eternal-wiecw', 'axle-bjpov', 5, 'yolov11'
DATASET_INFO = {'titulo': 'axle', 'autor': 'eternal', 'url': 'https://universe.roboflow.com/eternal-wiecw/axle-bjpov',
                'licencia': 'No indicada en la página de Roboflow Universe (se cita al autor)',
                'workspace': RF_WORKSPACE, 'proyecto': RF_PROYECTO, 'version': RF_VERSION, 'formato': RF_FORMATO}
CLASE_OBJETIVO = 'axle'      # única clase que se conserva (se busca por nombre en data.yaml)
NOMBRE_CLASE_POSE = 'llanta' # nombre de la clase del modelo pose (el servidor no filtra por nombre)
EPOCHS, IMGSZ, BATCH, PATIENCE, TOPE_HORAS = 50, 640, 16, 10, 0.5
REPO_URL = 'https://github.com/jayutomas623/sistema-peaje-ia'
DIR_DRIVE = '/content/drive/MyDrive/peaje-ia'
DIR_FINAL = '/content/m3_dataset'
M3_PT = '/content/m3_llantas_pose.pt'

In [ ]:
# === UTILIDADES COLAB (API key, Drive, servidor de inferencia del repo) ===
def obtener_api_key():
    # 1) Secreto de Colab 'ROBOFLOW_API_KEY' (🔑 en la barra lateral, con acceso habilitado para este notebook)
    # 2) Si no existe, se pide con getpass (no queda escrita en el notebook ni en el repositorio)
    try:
        from google.colab import userdata
        clave = userdata.get('ROBOFLOW_API_KEY')
        if clave:
            return clave
    except Exception as e:
        print('Secreto ROBOFLOW_API_KEY no disponible (' + type(e).__name__ + '): se pedirá a mano.')
    from getpass import getpass
    return getpass('Pegue su ROBOFLOW_API_KEY (no se guarda): ').strip()

def numero_version(v):
    # Objeto Version del SDK de Roboflow -> número entero (tolera distintas versiones del SDK)
    for valor in (getattr(v, 'version', None), getattr(v, 'id', None), v):
        try:
            return int(str(valor).rstrip('/').split('/')[-1])
        except (TypeError, ValueError):
            continue
    return -1

def descargar_roboflow(workspace, proyecto, version, formato):
    from roboflow import Roboflow
    rf = Roboflow(api_key=obtener_api_key())
    p = rf.workspace(workspace).project(proyecto)
    try:
        versiones = sorted(numero_version(v) for v in p.versions())
    except Exception as e:
        versiones = []; print('No se pudo listar versiones:', e)
    print(f'{workspace}/{proyecto} · versiones disponibles: {versiones} · se usa v{version}')
    if versiones and max(versiones) != version:
        print(f'⚠️  La última versión publicada es v{max(versiones)}. Se mantiene v{version} (la documentada en '
              f'entrenamiento/DATASETS.md); si cambia RF_VERSION, actualice también DATASETS.md.')
    return p.version(version).download(formato).location

def clonar_repo(destino='/content/sistema-peaje-ia'):
    if not os.path.isdir(destino):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, destino], check=True)
    return destino

def importar_servidor(repo_dir, **env):
    # Importa inferencia/peaje_ia (el MISMO código que corre en RunPod) con las variables de entorno dadas
    for k, v in env.items():
        os.environ[k] = str(v)
    for m in [m for m in sys.modules if m == 'peaje_ia' or m.startswith('peaje_ia.')]:
        del sys.modules[m]                      # recarga config.py con las nuevas rutas de pesos
    ruta = os.path.join(repo_dir, 'inferencia')
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

def guardar_en_drive(archivos, subcarpeta=''):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive no montado:', e); return
    for a in archivos:
        if not os.path.exists(a):
            print('  (no existe, se omite)', a); continue
        # pesos -> MyDrive/peaje-ia/pesos/ · métricas, CSV y gráficos -> MyDrive/peaje-ia/<subcarpeta>/
        dest = os.path.join(DIR_DRIVE, 'pesos' if a.endswith('.pt') else subcarpeta)
        os.makedirs(dest, exist_ok=True)
        shutil.copy(a, dest); print('  ->', os.path.join(dest, os.path.basename(a)))

def tope_de_tiempo(horas):
    # Tope de seguridad para Colab: detiene el entrenamiento al terminar la época en que se supera `horas`.
    # (No se usa el argumento `time=` de Ultralytics porque recalcula `epochs` para llenar el tiempo.)
    inicio = time.time()
    def cb(trainer):
        if time.time() - inicio > horas * 3600:
            print(f'⏱️  Tope de {horas} h alcanzado: se detiene tras la época {trainer.epoch + 1}.')
            trainer.stop = True
    return cb

In [ ]:
# === FUNCIONES M3: caja -> 5 keypoints (probadas localmente con datos sintéticos) ===
import os, glob, shutil
from collections import Counter

KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']   # mismo orden que el servidor
FLIP_IDX = [0, 1, 2, 4, 3]                                                  # flip horizontal: izquierda <-> derecha

def normalizar_nombre(n):
    return str(n).strip().lower().replace('_', '-').replace(' ', '-')

def nombres_de_yaml(cfg):
    n = cfg['names']
    return list(n) if isinstance(n, (list, tuple)) else [n[k] for k in sorted(n, key=int)]

def indice_clase(nombres, objetivo):
    ids = [i for i, n in enumerate(nombres) if normalizar_nombre(n) == normalizar_nombre(objetivo)]
    if len(ids) != 1:
        raise ValueError(f'No se encontró una única clase "{objetivo}" en data.yaml. names = {nombres}')
    return ids[0]

def caja_de_tokens(t):
    # 'cls cx cy w h' (detección) o 'cls x1 y1 x2 y2 ...' (polígono -> caja envolvente). Todo normalizado.
    if len(t) == 5:
        cx, cy, w, h = map(float, t[1:5])
        return cx, cy, w, h
    if len(t) >= 7 and (len(t) - 1) % 2 == 0:
        xs, ys = [float(v) for v in t[1::2]], [float(v) for v in t[2::2]]
        return (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
    return None

def _recortar(v):
    return min(max(v, 0.0), 1.0)

def caja_a_keypoints(cx, cy, w, h):
    # 5 keypoints geométricos de una llanta a partir de su caja (orden del notebook original y del servidor):
    # 0 centro (cx,cy) · 1 contacto (cx,cy+h/2) · 2 superior (cx,cy-h/2) · 3 izquierda (cx-w/2,cy) · 4 derecha (cx+w/2,cy)
    pts = [(cx, cy), (cx, cy + h / 2), (cx, cy - h / 2), (cx - w / 2, cy), (cx + w / 2, cy)]
    return [(_recortar(x), _recortar(y), 2) for x, y in pts]

def linea_pose(linea, id_objetivo):
    # -> (línea YOLO-pose 'cls cx cy w h x1 y1 v1 ... x5 y5 v5' | None, motivo)
    t = linea.split()
    if not t:
        return None, 'vacia'
    try:
        c = int(float(t[0]))
    except ValueError:
        return None, 'invalida'
    if c != id_objetivo:
        return None, 'otra_clase'
    caja = caja_de_tokens(t)
    if caja is None or caja[2] <= 0 or caja[3] <= 0:
        return None, 'invalida'
    cx, cy, w, h = caja
    valores = [cx, cy, w, h] + [v for kp in caja_a_keypoints(cx, cy, w, h) for v in kp]
    return '0 ' + ' '.join(str(v) if isinstance(v, int) else f'{v:.6f}' for v in valores), 'ok'

def leer_etiquetas(raiz, split):
    for f in sorted(glob.glob(os.path.join(raiz, split, 'labels', '*.txt'))):
        with open(f, encoding='utf-8') as fh:
            yield f, fh.read().splitlines()

def contar_por_clase(raiz, splits, nombres):
    res = {s: Counter() for s in splits}
    for s in splits:
        for _, lineas in leer_etiquetas(raiz, s):
            for l in lineas:
                t = l.split()
                if t:
                    i = int(float(t[0]))
                    res[s][nombres[i] if 0 <= i < len(nombres) else f'id{i}'] += 1
    return res

def contar_cajas(ruta_lbl, id_objetivo):
    # GT de ejes de una imagen = número de cajas de la clase objetivo
    if not os.path.exists(ruta_lbl):
        return 0
    with open(ruta_lbl, encoding='utf-8') as fh:
        return sum(1 for l in fh.read().splitlines() if l.split() and int(float(l.split()[0])) == id_objetivo)

def _enlazar_o_copiar(origen, destino):
    try:
        os.symlink(os.path.abspath(origen), destino, target_is_directory=True)
    except (OSError, NotImplementedError):
        shutil.copytree(origen, destino)

def escribir_dataset_pose(origen, destino, splits, id_objetivo):
    motivos, resumen = {s: Counter() for s in splits}, {}
    shutil.rmtree(destino, ignore_errors=True)
    for s in splits:
        dir_img = os.path.join(origen, s, 'images')
        os.makedirs(os.path.join(destino, s, 'labels'))
        _enlazar_o_copiar(dir_img, os.path.join(destino, s, 'images'))
        n_img = n_fondo = 0
        for nombre in sorted(os.listdir(dir_img)):
            base = os.path.splitext(nombre)[0]
            lbl = os.path.join(origen, s, 'labels', base + '.txt')
            lineas = []
            if os.path.exists(lbl):
                with open(lbl, encoding='utf-8') as fh:
                    lineas = fh.read().splitlines()
            nuevas = []
            for l in lineas:
                nl, m = linea_pose(l, id_objetivo)
                if m != 'vacia':
                    motivos[s][m] += 1
                if nl:
                    nuevas.append(nl)
            with open(os.path.join(destino, s, 'labels', base + '.txt'), 'w', encoding='utf-8') as fh:
                fh.write('\n'.join(nuevas) + ('\n' if nuevas else ''))
            n_img += 1
            n_fondo += not nuevas
        resumen[s] = {'imagenes': n_img, 'imagenes_fondo': n_fondo}
    return motivos, resumen

def yaml_pose(destino, splits, nombre_clase='llanta'):
    d = {'path': os.path.abspath(destino), 'train': 'train/images', 'val': 'valid/images',
         'kpt_shape': [5, 3], 'flip_idx': FLIP_IDX, 'nc': 1, 'names': {0: nombre_clase}}
    if 'test' in splits:
        d['test'] = 'test/images'
    return d

def metricas_conteo(gt, pred):
    # MAE, sesgo, % exacto y % dentro de ±1 entre listas de enteros
    assert len(gt) == len(pred) and gt, 'listas vacías o de distinto tamaño'
    err = [p - g for g, p in zip(gt, pred)]
    n = len(err)
    return {'n': n, 'MAE': sum(abs(e) for e in err) / n, 'sesgo': sum(err) / n,
            'exacto_pct': 100 * sum(e == 0 for e in err) / n, 'dentro_1_pct': 100 * sum(abs(e) <= 1 for e in err) / n}

## 3. Descarga del dataset, clases y conteo por split

In [ ]:
DS_DIR = descargar_roboflow(RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO)
SPLITS = [s for s in ('train', 'valid', 'test') if os.path.isdir(f'{DS_DIR}/{s}/labels')]
print(open(f'{DS_DIR}/data.yaml').read())
NOMBRES_ORIGEN = nombres_de_yaml(yaml.safe_load(open(f'{DS_DIR}/data.yaml')))
ID_AXLE = indice_clase(NOMBRES_ORIGEN, CLASE_OBJETIVO)     # se detiene con los names si no existe
print(f'names: {NOMBRES_ORIGEN} · "{CLASE_OBJETIVO}" = id {ID_AXLE} · splits: {SPLITS}')
CONTEO = {s: dict(c) for s, c in contar_por_clase(DS_DIR, SPLITS, NOMBRES_ORIGEN).items()}
pd.DataFrame(CONTEO).fillna(0).astype(int)

### 3.1 ¿Las cajas `axle` son llantas?
La conversión supone que cada caja `axle` encierra **una llanta** vista de lado (proporción ancho/alto ≈ 1).
Se mide la proporción en píxeles sobre una muestra y se dibujan ejemplos.

In [ ]:
props = []
for f, lineas in list(leer_etiquetas(DS_DIR, 'train'))[:400]:
    base = os.path.splitext(os.path.basename(f))[0]
    rutas = glob.glob(f'{DS_DIR}/train/images/{glob.escape(base)}.*')
    if not rutas: continue
    H, W = cv2.imread(rutas[0]).shape[:2]
    for l in lineas:
        t = l.split()
        if t and int(float(t[0])) == ID_AXLE and caja_de_tokens(t):
            _, _, w, h = caja_de_tokens(t); props.append((w * W) / max(1e-6, h * H))
PROP_MEDIANA = float(np.median(props)) if props else None
print(f'Proporción ancho/alto (px) de cajas axle: mediana = {PROP_MEDIANA} · n = {len(props)}')
if PROP_MEDIANA and not 0.5 <= PROP_MEDIANA <= 1.6:
    print('⚠️  Las cajas no parecen llantas individuales: revise los ejemplos antes de entrenar.')
plt.hist(props, bins=40); plt.xlabel('ancho / alto'); plt.title('Cajas axle'); plt.show()

## 4. Conversión caja → 5 keypoints (formato YOLO-pose)

In [ ]:
MOTIVOS, RESUMEN_IMGS = escribir_dataset_pose(DS_DIR, DIR_FINAL, SPLITS, ID_AXLE)
DATA_FINAL = f'{DIR_FINAL}/data.yaml'
yaml.safe_dump(yaml_pose(DIR_FINAL, SPLITS, NOMBRE_CLASE_POSE), open(DATA_FINAL, 'w'), sort_keys=False)
print(open(DATA_FINAL).read())
display(pd.DataFrame(MOTIVOS).fillna(0).astype(int)); display(pd.DataFrame(RESUMEN_IMGS))

COLORES = [(94, 63, 244), (0, 200, 0), (0, 200, 255), (255, 120, 0), (255, 0, 200)]
ejemplos = [f for f, l in leer_etiquetas(DIR_FINAL, 'train') if l][:3]
fig, ax = plt.subplots(1, len(ejemplos), figsize=(6 * len(ejemplos), 5))
for a, f in zip(np.atleast_1d(ax), ejemplos):
    base = os.path.splitext(os.path.basename(f))[0]
    img = cv2.imread(glob.glob(f'{DIR_FINAL}/train/images/{glob.escape(base)}.*')[0]); H, W = img.shape[:2]
    for l in open(f).read().split('\n'):
        v = l.split()
        if len(v) != 20: continue
        cx, cy, bw, bh = map(float, v[1:5])
        cv2.rectangle(img, (int((cx - bw / 2) * W), int((cy - bh / 2) * H)), (int((cx + bw / 2) * W), int((cy + bh / 2) * H)), (255, 255, 255), 2)
        for j in range(5):
            cv2.circle(img, (int(float(v[5 + 3 * j]) * W), int(float(v[6 + 3 * j]) * H)), 6, COLORES[j], -1)
    a.imshow(img[:, :, ::-1]); a.axis('off')
plt.suptitle(' · '.join(f'{j} {n}' for j, n in enumerate(KPT_NOMBRES))); plt.show()

## 5. Entrenamiento YOLO11s-pose
`epochs=50`, `patience=10`, `imgsz=640`. Tope de **0.5 h** con *callback* (no `time=`, que recalcula `epochs`).

In [ ]:
modelo = YOLO('yolo11s-pose.pt')        # el cabezal de 17 keypoints (humano) se adapta a 5
modelo.add_callback('on_fit_epoch_end', tope_de_tiempo(TOPE_HORAS))
t0 = time.time()
modelo.train(data=DATA_FINAL, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, fliplr=0.5,
             project='/content/runs/pose', name='m3', exist_ok=True, device=DEVICE, plots=True)
TIEMPO_MIN = round((time.time() - t0) / 60, 1)
RUN_DIR = str(modelo.trainer.save_dir)
EPOCAS_REALIZADAS = len(pd.read_csv(f'{RUN_DIR}/results.csv'))
print(f'Entrenamiento: {TIEMPO_MIN} min · {EPOCAS_REALIZADAS} épocas · {RUN_DIR}')

## 6. mAP de pose y de caja

In [ ]:
best = YOLO(f'{RUN_DIR}/weights/best.pt')
def metricas(split):
    v = best.val(data=DATA_FINAL, split=split, imgsz=IMGSZ, device=DEVICE,
                 project='/content/runs/pose', name=f'm3_{split}', exist_ok=True)
    return {'mAP50_pose': float(v.pose.map50), 'mAP50_95_pose': float(v.pose.map),
            'mAP50_box': float(v.box.map50), 'mAP50_95_box': float(v.box.map)}
MET = {s: metricas(s) for s in ('valid', 'test') if s in SPLITS}
display(pd.DataFrame(MET).T.round(4))
shutil.copy(f'{RUN_DIR}/weights/best.pt', M3_PT)

## 7. MAE del conteo de ejes (split test) con la regla del servidor
* **GT** = número de cajas `axle` de la imagen (etiquetas originales del dataset).
* **Predicción** = `DetectorEjes.inferir` de `inferencia/peaje_ia/m3_ejes.py` cargando `m3_llantas_pose.pt`
  (limpieza de llantas contenidas, lado cercano, separación entre ejes y **mínimo 2 ejes**). Así se comprueba
  además que el `.pt` carga en el servidor sin cambios.
* Si el servidor responde `ejes = None` (vista frontal/posterior o sin llantas), el backend cobra la tarifa
  base de 2 ejes: aquí se evalúa como **2** y se informa cuántas veces ocurrió.
* Se excluyen las imágenes sin ninguna caja `axle` (no hay ejes que contar).

In [ ]:
REPO_DIR = clonar_repo()
importar_servidor(REPO_DIR, M3_PESOS=M3_PT, M3_USAR_YOLOE='0', DEVICE='auto')
from peaje_ia.m3_ejes import DetectorEjes
det = DetectorEjes()
assert det.modo == 'entrenado', f'El servidor no cargó el modelo entrenado: {det.nombre}'
print('Servidor:', det.nombre)

SPLIT_EJES = 'test' if 'test' in SPLITS else 'valid'
filas = []
for ruta in sorted(glob.glob(f'{DS_DIR}/{SPLIT_EJES}/images/*')):
    base = os.path.splitext(os.path.basename(ruta))[0]
    gt = contar_cajas(f'{DS_DIR}/{SPLIT_EJES}/labels/{base}.txt', ID_AXLE)
    img = cv2.imread(ruta)
    if img is None: continue
    r = det.inferir(img, None)
    filas.append({'imagen': os.path.basename(ruta), 'ejes_gt': gt, 'ejes_servidor': r['ejes'],
                  'ejes_evaluados': r['ejes'] if r['ejes'] is not None else 2, 'vista': r['vista'],
                  'llantas_detectadas': len(r['llantas']), 'nota': r['nota']})
df_ej = pd.DataFrame(filas)
df_ej.to_csv('/content/m3_eval_ejes.csv', index=False)
eval_ = df_ej[df_ej.ejes_gt >= 1]
EJES = metricas_conteo(eval_.ejes_gt.tolist(), eval_.ejes_evaluados.tolist())
EJES_GT2 = metricas_conteo(*[df_ej[df_ej.ejes_gt >= 2][c].tolist() for c in ('ejes_gt', 'ejes_evaluados')]) \
    if (df_ej.ejes_gt >= 2).any() else None
EJES.update({'split': SPLIT_EJES, 'excluidas_sin_axle': int((df_ej.ejes_gt == 0).sum()),
             'servidor_devolvio_none': int(eval_.ejes_servidor.isna().sum())})
print(json.dumps({'todas_gt>=1': EJES, 'solo_gt>=2': EJES_GT2}, indent=2))
display(pd.crosstab(eval_.ejes_gt, eval_.ejes_evaluados))

## 8. Visualización (estilo imagen referencial M3)

In [ ]:
for ruta in sorted(glob.glob(f'{DS_DIR}/{SPLIT_EJES}/images/*'))[:4]:
    img = cv2.imread(ruta); r = det.inferir(img, None)
    puntos = sorted([l['kps'][0, :2] for l in r['llantas']], key=lambda p: p[0])
    for i, p in enumerate(puntos, 1):
        c = tuple(map(int, p)); cv2.circle(img, c, 12, (255, 255, 255), -1); cv2.circle(img, c, 9, (94, 63, 244), -1)
        cv2.putText(img, f'K{i}', (c[0] - 15, c[1] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (94, 63, 244), 2)
    for a, b in zip(puntos, puntos[1:]):
        cv2.line(img, tuple(map(int, a)), tuple(map(int, b)), (94, 63, 244), 2)
    gt = contar_cajas(f'{DS_DIR}/{SPLIT_EJES}/labels/{os.path.splitext(os.path.basename(ruta))[0]}.txt', ID_AXLE)
    plt.figure(figsize=(10, 6)); plt.imshow(img[:, :, ::-1]); plt.axis('off')
    plt.title(f"servidor: {r['ejes']} ejes ({r['vista']}) · GT: {gt} cajas axle"); plt.show()

## 9. `m3_metricas.json` y copia en Google Drive

In [ ]:
metricas_json = {
    'modelo': 'yolo11s-pose fine-tuning', 'archivo': 'm3_llantas_pose.pt',
    'keypoints_origen': 'derivados geométricamente de cajas del dataset eternal/axle',
    'keypoints': KPT_NOMBRES, 'kpt_shape': [5, 3], 'flip_idx': FLIP_IDX, 'clase_modelo': NOMBRE_CLASE_POSE,
    'dataset': DATASET_INFO, 'clase_usada': CLASE_OBJETIVO, 'clases_origen': NOMBRES_ORIGEN,
    'instancias_por_split': CONTEO, 'lineas_por_motivo': {s: dict(c) for s, c in MOTIVOS.items()},
    'imagenes': RESUMEN_IMGS, 'proporcion_ancho_alto_mediana': PROP_MEDIANA,
    'entrenamiento': {'epochs': EPOCHS, 'epocas_realizadas': EPOCAS_REALIZADAS, 'patience': PATIENCE,
                      'imgsz': IMGSZ, 'batch': BATCH, 'tope_horas': TOPE_HORAS, 'tiempo_min': TIEMPO_MIN},
    'metricas': MET,
    'conteo_ejes': {'regla': 'DetectorEjes del servidor (mínimo 2 ejes; None se evalúa como 2)',
                    'gt': 'número de cajas axle por imagen', 'gt_mayor_igual_1': EJES, 'gt_mayor_igual_2': EJES_GT2,
                    'csv': 'm3_eval_ejes.csv'},
}
json.dump(metricas_json, open('/content/m3_metricas.json', 'w'), indent=2, ensure_ascii=False)
guardar_en_drive([M3_PT, '/content/m3_metricas.json', '/content/m3_eval_ejes.csv', f'{RUN_DIR}/results.png'],
                 subcarpeta='metricas/m3')
print(json.dumps({'metricas': MET, 'MAE_ejes': EJES['MAE']}, indent=2))